In [ ]:
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit


PROJECT_ROOT = (
    Path.home()
    / "khacbinh"
    / "badminton-stroke"
)

ARCHIVE_PATH = PROJECT_ROOT / "data.zip"
DATA_DIR = PROJECT_ROOT / "data"
WORK_ROOT = PROJECT_ROOT / "processed"
MANIFEST_DIR = WORK_ROOT / "manifests"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

print("Project:", PROJECT_ROOT)
print("Archive:", ARCHIVE_PATH)
print("Data:", DATA_DIR)
print("Processed:", WORK_ROOT)

assert PROJECT_ROOT.exists(), PROJECT_ROOT


Project: /home/dutai/khacbinh/pbl5_doantotnghiep
Archive: /home/dutai/khacbinh/pbl5_doantotnghiep/data.zip
Data: /home/dutai/khacbinh/pbl5_doantotnghiep/data
Processed: /home/dutai/khacbinh/pbl5_doantotnghiep/processed


## Tải `data.zip` từ Google Drive

File phải được chia sẻ ở chế độ **Anyone with the link**. Dán link chia sẻ riêng của file `data.zip` vào cell bên dưới.


In [ ]:
from zipfile import is_zipfile

import gdown



GOOGLE_DRIVE_DATA_URL = (
    "PASTE_DATA_ZIP_SHARE_LINK_HERE"
)

if ARCHIVE_PATH.exists() and is_zipfile(ARCHIVE_PATH):
    print("ZIP đã tồn tại và hợp lệ:", ARCHIVE_PATH)

else:
    assert (
        "PASTE_DATA_ZIP_SHARE_LINK_HERE"
        not in GOOGLE_DRIVE_DATA_URL
    ), (
        "Hãy dán link chia sẻ của file data.zip "
        "vào GOOGLE_DRIVE_DATA_URL."
    )

    temporary_archive = (
        ARCHIVE_PATH.parent
        / f"{ARCHIVE_PATH.name}.part"
    )

    print("Đang tải data.zip từ Google Drive...")

    downloaded_path = gdown.download(
        url=GOOGLE_DRIVE_DATA_URL,
        output=str(temporary_archive),
        quiet=False,
        fuzzy=True,
        resume=True,
    )

    assert downloaded_path is not None, (
        "Google Drive không cho tải file. "
        "Hãy kiểm tra quyền Anyone with the link."
    )

    assert is_zipfile(temporary_archive), (
        f"File tải về không phải ZIP hợp lệ: "
        f"{temporary_archive}"
    )

    temporary_archive.replace(
        ARCHIVE_PATH
    )

    print("Đã tải xong:", ARCHIVE_PATH)

print(
    "Dung lượng ZIP:",
    f"{ARCHIVE_PATH.stat().st_size / (1024 ** 3):.2f} GB",
)


In [2]:
dataset_candidates = [
    path
    for path in DATA_DIR.rglob(
        "VideoBadminton_Dataset"
    )
    if path.is_dir()
]

if not dataset_candidates:
    print("Đang giải nén data.zip...")

    with ZipFile(ARCHIVE_PATH, "r") as archive:
        archive.extractall(DATA_DIR)

    dataset_candidates = [
        path
        for path in DATA_DIR.rglob(
            "VideoBadminton_Dataset"
        )
        if path.is_dir()
    ]

print(
    "Số thư mục VideoBadminton_Dataset:",
    len(dataset_candidates),
)

for candidate in dataset_candidates:
    print("-", candidate)


Số thư mục VideoBadminton_Dataset: 1
- /home/dutai/khacbinh/pbl5_doantotnghiep/data/data/VideoBadminton_Dataset


In [3]:
assert len(dataset_candidates) == 1, (
    "Cần tìm thấy đúng một thư mục "
    f"VideoBadminton_Dataset: {dataset_candidates}"
)

DATA_ROOT = dataset_candidates[0]

print("DATA_ROOT:", DATA_ROOT)


DATA_ROOT: /home/dutai/khacbinh/pbl5_doantotnghiep/data/data/VideoBadminton_Dataset


### Kiểm tra các lớp và video

In [4]:
class_dirs = sorted(
    path
    for path in DATA_ROOT.iterdir()
    if path.is_dir()
)

video_paths = sorted(
    DATA_ROOT.rglob("*.mp4")
)

print("Số lớp:", len(class_dirs))
print("Số video:", len(video_paths))

for class_dir in class_dirs:
    num_videos = len(
        list(class_dir.glob("*.mp4"))
    )

    print(
        f"{class_dir.name:35s}: "
        f"{num_videos}"
    )

assert len(class_dirs) == 18
assert len(video_paths) > 0

Số lớp: 18
Số video: 7822
00_Short Serve                     : 881
01_Cross Court Flight              : 167
02_Lift                            : 758
03_Tap Smash                       : 109
04_Block                           : 316
05_Drop Shot                       : 812
06_Push Shot                       : 401
07_Transitional Slice              : 123
08_Cut                             : 585
09_Rush Shot                       : 113
10_Defensive Clear                 : 109
11_Defensive Drive                 : 111
12_Clear                           : 945
13_Long Serve                      : 991
14_Smash                           : 822
15_Flat Shot                       : 327
16_Rear Court Flat Drive           : 121
17_Short Flat Shot                 : 131


## Tạo danh sách lớp

In [5]:
class_records = []

for class_dir in class_dirs:
    class_id_text, class_name = (
        class_dir.name.split(
            "_",
            maxsplit=1,
        )
    )

    class_records.append({
        "class_id": int(class_id_text),
        "class_name": class_name,
        "class_dir": class_dir.name,
    })

classes_df = (
    pd.DataFrame(class_records)
    .sort_values("class_id")
    .reset_index(drop=True)
)

assert (
    classes_df["class_id"].tolist()
    == list(range(18))
)

classes_df

,class_id,class_name,class_dir
0,0,Short Serve,00_Short Serve
1,1,Cross Court Flight,01_Cross Court Flight
2,2,Lift,02_Lift
3,3,Tap Smash,03_Tap Smash
4,4,Block,04_Block
5,5,Drop Shot,05_Drop Shot
6,6,Push Shot,06_Push Shot
7,7,Transitional Slice,07_Transitional Slice
8,8,Cut,08_Cut
9,9,Rush Shot,09_Rush Shot


### Đọc thông tin từ tên video

In [6]:
import re


FILENAME_PATTERN = re.compile(
    r"^"
    r"(?P<date>\d{4}-\d{2}-\d{2})_"
    r"(?P<time>\d{2}-\d{2}-\d{2})_"
    r"dataset_"
    r"(?P<set_name>set\d+)_"
    r"(?P<stroke_index>\d+)_"
    r"(?P<start_frame>\d+)_"
    r"(?P<end_frame>\d+)_"
    r"(?P<player>[A-Za-z]+)_"
    r"(?P<filename_class_id>\d+)"
    r"$"
)


def parse_video_path(video_path: Path) -> dict:
    match = FILENAME_PATTERN.match(
        video_path.stem
    )

    if match is None:
        raise ValueError(
            f"Tên file không hợp lệ: "
            f"{video_path.name}"
        )

    metadata = match.groupdict()

    folder_class_id, folder_class_name = (
        video_path.parent.name.split(
            "_",
            maxsplit=1,
        )
    )

    start_frame = int(
        metadata["start_frame"]
    )

    end_frame = int(
        metadata["end_frame"]
    )

    return {
        "clip_id": video_path.stem,
        "filename": video_path.name,

        # Dùng relative_path để không phụ thuộc
        # đường dẫn Colab hoặc Kaggle.
        "relative_path": str(
            video_path.relative_to(DATA_ROOT)
        ),

        "video_path": str(video_path),

        # Các clip cùng phiên quay phải nằm
        # trong cùng một split.
        "source_id": (
            f"{metadata['date']}_"
            f"{metadata['time']}"
        ),

        "date": metadata["date"],
        "time": metadata["time"],
        "set_name": metadata["set_name"],

        "stroke_index": int(
            metadata["stroke_index"]
        ),

        "start_frame": start_frame,
        "end_frame": end_frame,

        "expected_frames": (
            end_frame - start_frame + 1
        ),

        "player": metadata["player"],

        "filename_class_id": int(
            metadata["filename_class_id"]
        ),

        "class_id": int(folder_class_id),
        "class_name": folder_class_name,

        "file_size_bytes": (
            video_path.stat().st_size
        ),
    }

In [7]:
records = []
parse_errors = []

for video_path in video_paths:
    try:
        records.append(
            parse_video_path(video_path)
        )

    except Exception as error:
        parse_errors.append({
            "video_path": str(video_path),
            "error": str(error),
        })

manifest_df = pd.DataFrame(records)
parse_errors_df = pd.DataFrame(parse_errors)

print("Tổng video:", len(video_paths))
print("Đọc thành công:", len(manifest_df))
print("Lỗi tên file:", len(parse_errors_df))

manifest_df.head()

Tổng video: 7822
Đọc thành công: 7822
Lỗi tên file: 0


,clip_id,filename,relative_path,video_path,source_id,date,time,set_name,stroke_index,start_frame,end_frame,expected_frames,player,filename_class_id,class_id,class_name,file_size_bytes
0,2022-08-30_18-00-09_dataset_set1_009_001424_00...,2022-08-30_18-00-09_dataset_set1_009_001424_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,9,1424,1452,29,A,0,0,Short Serve,113004
1,2022-08-30_18-00-09_dataset_set1_016_001916_00...,2022-08-30_18-00-09_dataset_set1_016_001916_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,16,1916,1941,26,A,0,0,Short Serve,106926
2,2022-08-30_18-00-09_dataset_set1_023_002458_00...,2022-08-30_18-00-09_dataset_set1_023_002458_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,23,2458,2481,24,A,0,0,Short Serve,99961
3,2022-08-30_18-00-09_dataset_set1_051_004045_00...,2022-08-30_18-00-09_dataset_set1_051_004045_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,51,4045,4071,27,A,0,0,Short Serve,107765
4,2022-08-30_18-00-09_dataset_set1_053_004370_00...,2022-08-30_18-00-09_dataset_set1_053_004370_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,53,4370,4394,25,A,0,0,Short Serve,99362


## Kiểm tra và lọc dữ liệu lỗi 

In [8]:
manifest_df["has_valid_frame_range"] = (
    manifest_df["expected_frames"] > 0
)

manifest_df["is_non_empty"] = (
    manifest_df["file_size_bytes"] > 0
)

manifest_df["has_standard_player_code"] = (
    manifest_df["player"].isin(["A", "B"])
)

manifest_df["class_id_matches_filename"] = (
    manifest_df["class_id"]
    == manifest_df["filename_class_id"]
)

print(
    "Frame range không hợp lệ:",
    (~manifest_df[
        "has_valid_frame_range"
    ]).sum(),
)

print(
    "File rỗng:",
    (~manifest_df[
        "is_non_empty"
    ]).sum(),
)

print(
    "Player khác A/B:",
    (~manifest_df[
        "has_standard_player_code"
    ]).sum(),
)

print(
    "Class ID không khớp:",
    (~manifest_df[
        "class_id_matches_filename"
    ]).sum(),
)
manifest_df["is_usable"] = (
    manifest_df["has_valid_frame_range"]
    & manifest_df["is_non_empty"]
)

clean_df = (
    manifest_df[
        manifest_df["is_usable"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Tổng clip:", len(manifest_df))
print("Clip sử dụng được:", len(clean_df))
print(
    "Clip bị loại:",
    len(manifest_df) - len(clean_df),
)

Frame range không hợp lệ: 1
File rỗng: 0
Player khác A/B: 10
Class ID không khớp: 25
Tổng clip: 7822
Clip sử dụng được: 7821
Clip bị loại: 1


## Chia train, validation và test theo phiên quay

In [9]:
def class_distribution(
    dataframe,
    class_ids,
):
    counts = (
        dataframe["class_id"]
        .value_counts()
        .reindex(
            class_ids,
            fill_value=0,
        )
        .sort_index()
        .to_numpy(dtype=np.float64)
    )

    return counts / max(
        counts.sum(),
        1,
    )


def choose_group_split(
    dataframe,
    test_size,
    group_column="source_id",
    num_trials=2000,
    random_seed=42,
):
    class_ids = sorted(
        dataframe["class_id"].unique()
    )

    overall_distribution = (
        class_distribution(
            dataframe,
            class_ids,
        )
    )

    best_score = float("inf")
    best_train_indices = None
    best_test_indices = None

    for trial in range(num_trials):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=(
                random_seed + trial
            ),
        )

        train_indices, test_indices = next(
            splitter.split(
                dataframe,
                groups=dataframe[
                    group_column
                ],
            )
        )

        train_part = dataframe.iloc[
            train_indices
        ]

        test_part = dataframe.iloc[
            test_indices
        ]

        size_error = abs(
            len(test_part) / len(dataframe)
            - test_size
        )

        train_distribution = (
            class_distribution(
                train_part,
                class_ids,
            )
        )

        test_distribution = (
            class_distribution(
                test_part,
                class_ids,
            )
        )

        distribution_error = (
            np.abs(
                train_distribution
                - overall_distribution
            ).mean()
            +
            np.abs(
                test_distribution
                - overall_distribution
            ).mean()
        )

        score = (
            size_error
            + distribution_error
        )

        if score < best_score:
            best_score = score
            best_train_indices = (
                train_indices
            )
            best_test_indices = (
                test_indices
            )

    return (
        dataframe.iloc[
            best_train_indices
        ].copy(),
        dataframe.iloc[
            best_test_indices
        ].copy(),
    )
train_val_df, test_df = (
    choose_group_split(
        dataframe=clean_df,
        test_size=0.15,
        random_seed=RANDOM_SEED,
    )
)

train_df, val_df = (
    choose_group_split(
        dataframe=train_val_df,
        test_size=0.15 / 0.85,
        random_seed=(
            RANDOM_SEED + 10_000
        ),
    )
)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat(
    [
        train_df,
        val_df,
        test_df,
    ],
    ignore_index=True,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Tổng:", len(split_df))


Train: 5465
Validation: 1176
Test: 1180
Tổng: 7821


## Lưu Manifest 

In [10]:
manifest_df.to_csv(
    MANIFEST_DIR
    / "all_videos_audit.csv",
    index=False,
)

split_df.to_csv(
    MANIFEST_DIR
    / "dataset_manifest.csv",
    index=False,
)

train_df.to_csv(
    MANIFEST_DIR / "train.csv",
    index=False,
)

val_df.to_csv(
    MANIFEST_DIR / "val.csv",
    index=False,
)

test_df.to_csv(
    MANIFEST_DIR / "test.csv",
    index=False,
)

classes_df.to_csv(
    MANIFEST_DIR / "classes.csv",
    index=False,
)

if not parse_errors_df.empty:
    parse_errors_df.to_csv(
        MANIFEST_DIR
        / "parse_errors.csv",
        index=False,
    )

print("Đã lưu tại:", MANIFEST_DIR)

for path in MANIFEST_DIR.iterdir():
    print("-", path.name)

Đã lưu tại: /home/dutai/khacbinh/pbl5_doantotnghiep/processed/manifests
- all_videos_audit.csv
- classes.csv
- dataset_manifest.csv
- test.csv
- train.csv
- val.csv


In [11]:
saved_manifest = pd.read_csv(
    MANIFEST_DIR
    / "dataset_manifest.csv"
)

print(saved_manifest.shape)
print(
    saved_manifest["split"]
    .value_counts()
)

saved_manifest.head()

(7821, 23)
split
train    5465
test     1180
val      1176
Name: count, dtype: int64


,clip_id,filename,relative_path,video_path,source_id,date,time,set_name,stroke_index,start_frame,...,filename_class_id,class_id,class_name,file_size_bytes,has_valid_frame_range,is_non_empty,has_standard_player_code,class_id_matches_filename,is_usable,split
0,2022-08-30_18-00-09_dataset_set1_009_001424_00...,2022-08-30_18-00-09_dataset_set1_009_001424_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,9,1424,...,0,0,Short Serve,113004,True,True,True,True,True,train
1,2022-08-30_18-00-09_dataset_set1_016_001916_00...,2022-08-30_18-00-09_dataset_set1_016_001916_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,16,1916,...,0,0,Short Serve,106926,True,True,True,True,True,train
2,2022-08-30_18-00-09_dataset_set1_023_002458_00...,2022-08-30_18-00-09_dataset_set1_023_002458_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,23,2458,...,0,0,Short Serve,99961,True,True,True,True,True,train
3,2022-08-30_18-00-09_dataset_set1_051_004045_00...,2022-08-30_18-00-09_dataset_set1_051_004045_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,51,4045,...,0,0,Short Serve,107765,True,True,True,True,True,train
4,2022-08-30_18-00-09_dataset_set1_053_004370_00...,2022-08-30_18-00-09_dataset_set1_053_004370_00...,00_Short Serve/2022-08-30_18-00-09_dataset_set...,/home/dutai/khacbinh/pbl5_doantotnghiep/data/d...,2022-08-30_18-00-09,2022-08-30,18-00-09,set1,53,4370,...,0,0,Short Serve,99362,True,True,True,True,True,train


# Joint Extraction



In [12]:
# Kiểm tra GPU và tiến trình đang sử dụng GPU.
!nvidia-smi


Tue Sep 29 09:32:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-16GB           On  |   00000000:01:00.0 Off |                    0 |
| N/A   45C    P0             40W /  300W |    5872MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
import onnxruntime as ort

# Nạp CUDA/cuDNN được cài bởi extra [cuda,cudnn].
ort.preload_dlls(directory="")

print("ONNX Runtime:", ort.__version__)
print(
    "Available providers:",
    ort.get_available_providers(),
)

assert ort.__version__ == "1.26.0", (
    "Notebook được kiểm thử với "
    "onnxruntime-gpu==1.26.0."
)

assert "CUDAExecutionProvider" in (
    ort.get_available_providers()
), "ONNX Runtime chưa nhận CUDA GPU."


ONNX Runtime: 1.26.0
Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


## Môi trường server

Môi trường đã được cài bằng `uv sync --frozen`. Không chạy `pip install` trong notebook.


In [14]:
from importlib.metadata import version

import cv2
import rtmlib

print("RTMLib:", version("rtmlib"))
print("OpenCV:", cv2.__version__)


RTMLib: 0.0.16
OpenCV: 5.0.0


In [15]:
# Xác nhận lại provider trước khi tải model.
print("ONNX Runtime:", ort.__version__)
print(
    "Available providers:",
    ort.get_available_providers(),
)


ONNX Runtime: 1.26.0
Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


## Đọc lại manifest và khôi phục đường dẫn video

In [16]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd


MANIFEST_PATH = (
    MANIFEST_DIR
    / "dataset_manifest.csv"
)

assert MANIFEST_PATH.exists(), (
    f"Không tìm thấy: {MANIFEST_PATH}"
)

manifest_df = pd.read_csv(
    MANIFEST_PATH
)

assert DATA_ROOT.exists(), DATA_ROOT

manifest_df["video_path"] = (
    manifest_df["relative_path"]
    .apply(
        lambda relative_path: str(
            DATA_ROOT / relative_path
        )
    )
)

if "clip_id" not in manifest_df.columns:
    manifest_df["clip_id"] = (
        manifest_df["filename"]
        .apply(
            lambda name: Path(name).stem
        )
    )

manifest_df = (
    manifest_df
    .sort_values(
        [
            "split",
            "class_id",
            "clip_id",
        ]
    )
    .reset_index(drop=True)
)

manifest_df["video_exists"] = (
    manifest_df["video_path"]
    .apply(
        lambda video_path: Path(
            video_path
        ).exists()
    )
)

print("Tổng clip:", len(manifest_df))
print(
    manifest_df["split"].value_counts()
)
print(
    "Video bị thiếu:",
    (~manifest_df["video_exists"]).sum(),
)

assert manifest_df["video_exists"].all()


Tổng clip: 7821
split
train    5465
test     1180
val      1176
Name: count, dtype: int64
Video bị thiếu: 0


## Khởi tạo RTMPose

In [17]:
from importlib.metadata import version

print("RTMLib:", version("rtmlib"))
print("ONNX Runtime:", ort.__version__)
print(
    "Providers:",
    ort.get_available_providers(),
)


RTMLib: 0.0.16
ONNX Runtime: 1.26.0
Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [18]:
from rtmlib import Body, PoseTracker

pose_model = PoseTracker(
    Body,
    mode="balanced",
    det_frequency=5,
    tracking=False,
    backend="onnxruntime",
    device="cuda",
    to_openpose=False,
)

detector_providers = (
    pose_model.det_model
    .session
    .get_providers()
)

pose_providers = (
    pose_model.pose_model
    .session
    .get_providers()
)

print("Detector:", detector_providers)
print("Pose:", pose_providers)

assert detector_providers[0] == (
    "CUDAExecutionProvider"
)

assert pose_providers[0] == (
    "CUDAExecutionProvider"
)


load /home/dutai/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend
load /home/dutai/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.onnx with onnxruntime backend
Detector: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Pose: ['CUDAExecutionProvider', 'CPUExecutionProvider']


2026-09-29 09:32:32.899247246 [W:onnxruntime:, session_state.cc:1367 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-09-29 09:32:32.899269922 [W:onnxruntime:, session_state.cc:1369 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2026-09-29 09:32:32.958147982 [W:onnxruntime:, session_state.cc:1367 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-09-29 09:32:32.958163684 [W:onnxruntime:, session_state.cc:1369 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [19]:
def pose_area(
    keypoints,
    scores,
    threshold=0.2,
):
    valid = scores >= threshold

    if valid.sum() < 4:
        return 0.0

    valid_points = keypoints[valid]

    width = (
        valid_points[:, 0].max()
        - valid_points[:, 0].min()
    )

    height = (
        valid_points[:, 1].max()
        - valid_points[:, 1].min()
    )

    return float(width * height)


def vertical_position(
    keypoints,
    scores,
    threshold=0.2,
):
    ankle_indices = [15, 16]

    valid_ankles = [
        index
        for index in ankle_indices
        if scores[index] >= threshold
    ]

    if valid_ankles:
        return float(
            keypoints[
                valid_ankles,
                1,
            ].mean()
        )

    valid = scores >= threshold

    if valid.any():
        return float(
            keypoints[valid, 1].mean()
        )

    return float("inf")


def select_two_players(
    keypoints,
    scores,
):
    if len(keypoints) < 2:
        return None, None

    areas = [
        pose_area(
            person_keypoints,
            person_scores,
        )
        for person_keypoints, person_scores
        in zip(keypoints, scores)
    ]

    # Chọn hai người có vùng pose lớn nhất.
    selected_indices = np.argsort(
        areas
    )[-2:]

    selected_keypoints = keypoints[
        selected_indices
    ]

    selected_scores = scores[
        selected_indices
    ]

    # Người ở phía trên sân trước,
    # người phía dưới sân sau.
    order = sorted(
        range(2),
        key=lambda index: vertical_position(
            selected_keypoints[index],
            selected_scores[index],
        ),
    )

    return (
        selected_keypoints[order],
        selected_scores[order],
    )

## Chuẩn hóa joint

In [20]:
def normalize_joints(
    keypoints,
    scores,
    threshold=0.2,
):
    result = np.zeros(
        (17, 2),
        dtype=np.float32,
    )

    valid = scores >= threshold

    if valid.sum() < 4:
        return result

    valid_points = keypoints[valid]

    min_xy = valid_points.min(axis=0)
    max_xy = valid_points.max(axis=0)

    center = (
        min_xy + max_xy
    ) / 2

    diagonal = np.linalg.norm(
        max_xy - min_xy
    )

    if diagonal <= 0:
        return result

    normalized = (
        keypoints - center
    ) / diagonal

    normalized[~valid] = 0.0

    number_of_joints = min(
        17,
        len(normalized),
    )

    result[:number_of_joints] = (
        normalized[:number_of_joints]
    )

    return result

In [21]:
def extract_video_joints(
    video_path,
    pose_model,
):
    # Xóa trạng thái từ video trước.
    pose_model.reset()

    capture = cv2.VideoCapture(
        str(video_path)
    )

    if not capture.isOpened():
        raise RuntimeError(
            f"Không mở được video: "
            f"{video_path}"
        )

    frames_joints = []

    try:
        while True:
            success, frame = capture.read()

            if not success:
                break

            keypoints, scores = (
                pose_model(frame)
            )

            keypoints = np.asarray(
                keypoints,
                dtype=np.float32,
            )

            scores = np.asarray(
                scores,
                dtype=np.float32,
            )

            (
                selected_keypoints,
                selected_scores,
            ) = select_two_players(
                keypoints,
                scores,
            )

            if selected_keypoints is None:
                frame_joints = np.zeros(
                    (2, 17, 2),
                    dtype=np.float32,
                )

            else:
                frame_joints = np.stack(
                    [
                        normalize_joints(
                            person_keypoints,
                            person_scores,
                        )
                        for (
                            person_keypoints,
                            person_scores,
                        )
                        in zip(
                            selected_keypoints,
                            selected_scores,
                        )
                    ],
                    axis=0,
                )

            frames_joints.append(
                frame_joints
            )

    finally:
        capture.release()

    if not frames_joints:
        return np.zeros(
            (0, 2, 17, 2),
            dtype=np.float32,
        )

    return np.stack(
        frames_joints,
        axis=0,
    )

## Benchmark trước khi chạy toàn bộ

In [22]:
import time


sample_row = manifest_df.iloc[0]

for run_index in range(2):
    start_time = time.time()

    sample_joints = (
        extract_video_joints(
            sample_row["video_path"],
            pose_model,
        )
    )

    elapsed = time.time() - start_time

    fps = (
        len(sample_joints) / elapsed
        if elapsed > 0
        else 0.0
    )

    print(
        f"Run {run_index + 1}: "
        f"{elapsed:.2f} giây, "
        f"{fps:.2f} FPS, "
        f"shape={sample_joints.shape}"
    )

Run 1: 0.81 giây, 24.68 FPS, shape=(20, 2, 17, 2)
Run 2: 0.47 giây, 42.76 FPS, shape=(20, 2, 17, 2)


In [23]:
JOINTS_ROOT = (
    WORK_ROOT
    / "joints_npy_balanced"
)

STATUS_PATH = (
    WORK_ROOT
    / "joint_extraction_status.csv"
)

JOINTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Joints:", JOINTS_ROOT)
print("Status:", STATUS_PATH)


Joints: /home/dutai/khacbinh/pbl5_doantotnghiep/processed/joints_npy_balanced
Status: /home/dutai/khacbinh/pbl5_doantotnghiep/processed/joint_extraction_status.csv


## Chạy toàn bộ dataset

In [24]:
from tqdm import tqdm


if STATUS_PATH.exists():
    old_status_df = pd.read_csv(
        STATUS_PATH
    )

    status_records = {
        row["clip_id"]: row
        for row in old_status_df.to_dict(
            orient="records"
        )
    }
else:
    status_records = {}


def get_output_path(row):
    class_directory = (
        f"{int(row.class_id):02d}_"
        f"{row.class_name}"
    )

    output_directory = (
        JOINTS_ROOT
        / row.split
        / class_directory
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    return (
        output_directory
        / f"{row.clip_id}_joints.npy"
    )


def save_status():
    if status_records:
        pd.DataFrame(
            status_records.values()
        ).to_csv(
            STATUS_PATH,
            index=False,
        )


In [25]:
for index, row in enumerate(
    tqdm(
        manifest_df.itertuples(
            index=False
        ),
        total=len(manifest_df),
        desc="Extracting joints",
    )
):
    output_path = get_output_path(row)

    # Tiếp tục an toàn sau khi SSH hoặc notebook bị ngắt.
    if output_path.exists():
        if row.clip_id not in status_records:
            existing_joints = np.load(
                output_path
            )

            valid_frames = np.any(
                existing_joints != 0,
                axis=(1, 2, 3),
            )

            status_records[row.clip_id] = {
                "clip_id": row.clip_id,
                "split": row.split,
                "class_id": int(row.class_id),
                "class_name": row.class_name,
                "status": "success",
                "num_frames": len(
                    existing_joints
                ),
                "valid_ratio": float(
                    valid_frames.mean()
                ),
                "output_path": str(
                    output_path
                ),
                "error": "",
            }

        continue

    try:
        joints = extract_video_joints(
            row.video_path,
            pose_model,
        )

        if len(joints) == 0:
            raise RuntimeError(
                "Video không có frame."
            )

        temporary_path = (
            output_path.parent
            / (
                output_path.stem
                + ".part.npy"
            )
        )

        np.save(
            temporary_path,
            joints.astype(np.float32),
        )

        temporary_path.replace(
            output_path
        )

        valid_frames = np.any(
            joints != 0,
            axis=(1, 2, 3),
        )

        status_records[row.clip_id] = {
            "clip_id": row.clip_id,
            "split": row.split,
            "class_id": int(row.class_id),
            "class_name": row.class_name,
            "status": "success",
            "num_frames": len(joints),
            "valid_ratio": float(
                valid_frames.mean()
            ),
            "output_path": str(
                output_path
            ),
            "error": "",
        }

    except Exception as error:
        status_records[row.clip_id] = {
            "clip_id": row.clip_id,
            "split": row.split,
            "class_id": int(row.class_id),
            "class_name": row.class_name,
            "status": "error",
            "num_frames": 0,
            "valid_ratio": 0.0,
            "output_path": str(
                output_path
            ),
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            ),
        }

        print(
            "\nLỗi:",
            row.clip_id,
            error,
        )

    if (index + 1) % 100 == 0:
        save_status()


save_status()

joint_files = list(
    JOINTS_ROOT.rglob(
        "*_joints.npy"
    )
)

status_df = pd.DataFrame(
    status_records.values()
)

print("Hoàn thành extraction.")
print("Số file joint:", len(joint_files))
print("Trạng thái:")
print(status_df["status"].value_counts())


Extracting joints:  42%|████▏     | 3313/7821 [00:00<00:00, 16618.39it/s]


Lỗi: 2022-08-31_18-50-39_dataset_set1_004_001116_001116_A_00 Không mở được video: /home/dutai/khacbinh/pbl5_doantotnghiep/data/data/VideoBadminton_Dataset/00_Short Serve/2022-08-31_18-50-39_dataset_set1_004_001116_001116_A_00.mp4

Lỗi: 2022-09-01_19-34-28_dataset_set1_038_002860_002860_A_04 Không mở được video: /home/dutai/khacbinh/pbl5_doantotnghiep/data/data/VideoBadminton_Dataset/04_Block/2022-09-01_19-34-28_dataset_set1_038_002860_002860_A_04.mp4


Extracting joints: 100%|██████████| 7821/7821 [00:00<00:00, 16760.76it/s]



Lỗi: 2022-08-31_19-19-47_dataset_set1_005_000606_000606_B_13 Không mở được video: /home/dutai/khacbinh/pbl5_doantotnghiep/data/data/VideoBadminton_Dataset/13_Long Serve/2022-08-31_19-19-47_dataset_set1_005_000606_000606_B_13.mp4
Hoàn thành extraction.
Số file joint: 7818
Trạng thái:
status
success    7818
error         3
Name: count, dtype: int64


In [26]:
from pathlib import Path
import numpy as np

JOINTS_ROOT = Path(
    "/home/dutai/khacbinh/pbl5_doantotnghiep/"
    "processed/joints_npy_balanced"
)

joint_files = list(JOINTS_ROOT.rglob("*_joints.npy"))

print("Số file:", len(joint_files))

invalid_files = []

for path in joint_files:
    try:
        joints = np.load(path, mmap_mode="r")

        # Dạng mong đợi: (số frame, 2 người, 17 joint, 2 tọa độ)
        valid = (
            joints.ndim == 4
            and joints.shape[1:] == (2, 17, 2)
            and joints.shape[0] > 0
        )

        if not valid:
            invalid_files.append((str(path), joints.shape))

    except Exception as error:
        invalid_files.append((str(path), str(error)))

print("File hợp lệ:", len(joint_files) - len(invalid_files))
print("File không hợp lệ:", len(invalid_files))

for item in invalid_files[:10]:
    print(item)

Số file: 7818
File hợp lệ: 7818
File không hợp lệ: 0
